In [3]:
# Sel 1: Buat struktur folder & instal library
!mkdir -p dataset models notebook
!pip install -q scikit-learn pandas numpy joblib streamlit
!npm install -g localtunnel

# Upload file kasus09_mode_choice.csv ke folder dataset/
from google.colab import files
import shutil, os

uploaded = files.upload()
for fn in uploaded.keys():
    shutil.move(fn, os.path.join("dataset", "kasus09_mode_choice.csv"))
print("Dataset siap di dataset/kasus09_mode_choice.csv")

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦
changed 22 packages in 2s
⠦
⠦3 packages are looking for funding
⠦  run `npm fund` for details
⠦

KeyboardInterrupt: 

In [4]:
%%writefile fitur.py
import pandas as pd
import numpy as np

MODES = [1, 2, 3, 4]
MODE_ATTRS = ['ttme', 'invc', 'invt', 'gc']

def audit_dan_bersihkan_long(df: pd.DataFrame) -> pd.DataFrame:
    """Audit tipe data, duplikat, missing values, dan validasi integritas target per individu."""
    df_clean = df.copy()

    # Hapus duplikat persis jika ada
    df_clean = df_clean.drop_duplicates()

    # Pastikan kolom identitas/kategori bertipe integer
    for col in ['individual', 'mode', 'choice']:
        df_clean[col] = df_clean[col].astype(int)

    # Validasi setiap individu hanya memiliki tepat 1 pilihan (choice == 1)
    choice_per_ind = df_clean.groupby('individual')['choice'].sum()
    valid_ids = choice_per_ind[choice_per_ind == 1].index
    df_clean = df_clean[df_clean['individual'].isin(valid_ids)].copy()

    return df_clean

def reshape_long_to_wide(df_long: pd.DataFrame) -> pd.DataFrame:
    """
    Mengubah dataset dari format long (4 baris per individu)
    menjadi format wide (1 baris per individu).
    """
    df_clean = audit_dan_bersihkan_long(df_long)

    # Ekstrak target (mode dengan choice == 1)
    target_df = (
        df_clean[df_clean['choice'] == 1][['individual', 'mode']]
        .rename(columns={'mode': 'target_mode'})
    )

    # Ekstrak karakteristik individu (konstan di keempat baris)
    indiv_df = df_clean.groupby('individual')[['hinc', 'psize']].first().reset_index()

    # Pivot atribut moda menjadi kolom terpisah per moda
    pivoted = df_clean.pivot(index='individual', columns='mode', values=MODE_ATTRS)
    pivoted.columns = [f"{attr}_mode{m}" for attr, m in pivoted.columns]
    pivoted = pivoted.reset_index()

    # Gabungkan menjadi satu tabel wide
    wide_df = indiv_df.merge(pivoted, on='individual').merge(target_df, on='individual')
    return wide_df

def buat_fitur_baru(X_df: pd.DataFrame) -> pd.DataFrame:
    """
    Membuat fitur rekayasa (feature engineering) tanpa kebocoran data (row-wise).
    Dipanggil setelah train-test split maupun saat inferensi di Streamlit.
    """
    X = X_df.copy()
    psize_safe = X['psize'].replace(0, 1)

    for m in MODES:
        # 1. Total waktu perjalanan (waktu terminal + waktu dalam kendaraan)
        X[f'total_time_mode{m}'] = X[f'ttme_mode{m}'] + X[f'invt_mode{m}']
        # 2. Biaya dalam kendaraan per orang
        X[f'invc_per_person_mode{m}'] = X[f'invc_mode{m}'] / psize_safe

    # 3. Selisih biaya gabungan & waktu total terhadap alternatif terbaik (termurah/tercepat)
    gc_cols = [f'gc_mode{m}' for m in MODES]
    time_cols = [f'total_time_mode{m}' for m in MODES]

    min_gc = X[gc_cols].min(axis=1)
    min_time = X[time_cols].min(axis=1)

    for m in MODES:
        X[f'gc_diff_min_mode{m}'] = X[f'gc_mode{m}'] - min_gc
        X[f'time_diff_min_mode{m}'] = X[f'total_time_mode{m}'] - min_time

    return X

def get_raw_feature_columns() -> list:
    """Daftar 18 kolom fitur mentah dalam format wide (sebelum rekayasa fitur)."""
    cols = ['hinc', 'psize']
    for attr in MODE_ATTRS:
        for m in MODES:
            cols.append(f"{attr}_mode{m}")
    return cols

Writing fitur.py


In [5]:
%%writefile latih.py
import os
import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

from fitur import reshape_long_to_wide, buat_fitur_baru, get_raw_feature_columns

DATA_PATH = os.path.join("dataset", "kasus09_mode_choice.csv")
MODEL_DIR = "models"
MODEL_PATH = os.path.join(MODEL_DIR, "model_mode_choice.joblib")

def main():
    # 1. Load & Reshape Long -> Wide
    df_long = pd.read_csv(DATA_PATH)
    wide_df = reshape_long_to_wide(df_long)

    # Buang kolom ID ('individual') dan pisahkan target ('target_mode')
    raw_cols = get_raw_feature_columns()
    X_raw = wide_df[raw_cols]
    y = wide_df['target_mode']

    # 2. Train-Test Split SEBELUM preprocessing (imputasi/scaling)
    X_train_raw, X_test_raw, y_train, y_test = train_test_split(
        X_raw, y, test_size=0.20, random_state=42, stratify=y
    )

    # 3. Terapkan Feature Engineering
    X_train = buat_fitur_baru(X_train_raw)
    X_test = buat_fitur_baru(X_test_raw)

    # 4. Definisikan Pipeline Model untuk Perbandingan
    pipelines = {
        "Logistic Regression": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(max_iter=1000, random_state=42))
        ]),
        "Random Forest": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("clf", RandomForestClassifier(n_estimators=200, random_state=42))
        ])
    }

    # 5. Pemilihan Model dengan Stratified Cross-Validation di Data Train
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_results = {}

    print("=== HASIL CROSS-VALIDATION (DATA TRAIN) ===")
    for name, pipe in pipelines.items():
        scores = cross_validate(
            pipe, X_train, y_train, cv=cv, scoring=["accuracy", "f1_macro"]
        )
        mean_acc = float(np.mean(scores["test_accuracy"]))
        std_acc = float(np.std(scores["test_accuracy"]))
        mean_f1 = float(np.mean(scores["test_f1_macro"]))
        std_f1 = float(np.std(scores["test_f1_macro"]))

        cv_results[name] = {
            "cv_accuracy_mean": mean_acc,
            "cv_accuracy_std": std_acc,
            "cv_f1_macro_mean": mean_f1,
            "cv_f1_macro_std": std_f1
        }
        print(f"{name:20s} | Accuracy: {mean_acc:.4f} (±{std_acc:.4f}) | Macro F1: {mean_f1:.4f} (±{std_f1:.4f})")

    # Pilih model terbaik berdasarkan Macro F1 CV (agar adil untuk kelas minoritas Moda 3)
    best_model_name = max(cv_results, key=lambda k: cv_results[k]["cv_f1_macro_mean"])
    best_pipeline = pipelines[best_model_name]
    print(f"\nModel Terpilih: {best_model_name}")

    # 6. Latih Model Terpilih pada Seluruh Data Train & Evaluasi SEKALI pada Data Test
    best_pipeline.fit(X_train, y_train)
    y_pred_test = best_pipeline.predict(X_test)

    test_acc = float(accuracy_score(y_test, y_pred_test))
    test_f1_macro = float(f1_score(y_test, y_pred_test, average="macro"))
    test_f1_weighted = float(f1_score(y_test, y_pred_test, average="weighted"))
    cm = confusion_matrix(y_test, y_pred_test, labels=[1, 2, 3, 4]).tolist()
    report_dict = classification_report(y_test, y_pred_test, output_dict=True)

    print("\n=== EVALUASI FINAL PADA DATA TEST (1x) ===")
    print(f"Test Accuracy    : {test_acc:.4f}")
    print(f"Test Macro F1    : {test_f1_macro:.4f}")
    print(f"Test Weighted F1 : {test_f1_weighted:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred_test))

    # 7. Simpan Pipeline + Metadata dengan Joblib
    os.makedirs(MODEL_DIR, exist_ok=True)
    artifact = {
        "pipeline": best_pipeline,
        "best_model_name": best_model_name,
        "raw_features": raw_cols,
        "engineered_features": list(X_train.columns),
        "classes": [1, 2, 3, 4],
        "class_distribution": y.value_counts().sort_index().to_dict(),
        "cv_results": cv_results,
        "test_metrics": {
            "accuracy": test_acc,
            "f1_macro": test_f1_macro,
            "f1_weighted": test_f1_weighted,
            "confusion_matrix": cm,
            "classification_report": report_dict
        },
        "default_sample": X_test_raw.iloc[0].to_dict()
    }
    joblib.dump(artifact, MODEL_PATH)
    print(f"\nArtifact berhasil disimpan ke: {MODEL_PATH}")

if __name__ == "__main__":
    main()

Writing latih.py


In [6]:
!python latih.py

=== HASIL CROSS-VALIDATION (DATA TRAIN) ===
Logistic Regression  | Accuracy: 0.7865 (±0.0716) | Macro F1: 0.7797 (±0.0922)
Random Forest        | Accuracy: 0.9048 (±0.0113) | Macro F1: 0.9043 (±0.0360)

Model Terpilih: Random Forest

=== EVALUASI FINAL PADA DATA TEST (1x) ===
Test Accuracy    : 0.9286
Test Macro F1    : 0.9357
Test Weighted F1 : 0.9281

Classification Report:
              precision    recall  f1-score   support

           1       1.00      0.82      0.90        11
           2       0.93      1.00      0.96        13
           3       1.00      1.00      1.00         6
           4       0.85      0.92      0.88        12

    accuracy                           0.93        42
   macro avg       0.94      0.93      0.94        42
weighted avg       0.93      0.93      0.93        42


Artifact berhasil disimpan ke: models/model_mode_choice.joblib


In [7]:
%%writefile app.py
import os
import joblib
import pandas as pd
import streamlit as st
from fitur import buat_fitur_baru, MODES

st.set_page_config(page_title="Prediksi Pilihan Moda Transportasi", layout="wide")

MODEL_PATH = os.path.join("models", "model_mode_choice.joblib")

@st.cache_resource
def load_artifact():
    if not os.path.exists(MODEL_PATH):
        return None
    return joblib.load(MODEL_PATH)

artifact = load_artifact()

st.title("Prediksi Pilihan Moda Transportasi (Kasus 9)")
st.markdown(
    "Aplikasi klasifikasi multikelas untuk memprediksi pilihan moda transportasi "
    "(**Moda 1 – 4**) berdasarkan karakteristik perjalanan dan atribut keempat alternatif moda."
)

if artifact is None:
    st.error("File model tidak ditemukan. Jalankan `python latih.py` terlebih dahulu.")
    st.stop()

pipeline = artifact["pipeline"]
default_sample = artifact["default_sample"]

tab_pred, tab_eval = st.tabs(["Simulasi Prediksi", "Metrik & Evaluasi Model"])

with tab_pred:
    st.subheader("1. Profil Pelancong")
    col_ind1, col_ind2 = st.columns(2)
    with col_ind1:
        hinc = st.number_input(
            "Pendapatan Rumah Tangga (hinc)",
            min_value=0.0, max_value=200.0,
            value=float(default_sample.get("hinc", 35.0)), step=1.0
        )
    with col_ind2:
        psize = st.number_input(
            "Ukuran Rombongan (psize)",
            min_value=1, max_value=15,
            value=int(default_sample.get("psize", 1)), step=1
        )

    st.subheader("2. Atribut Keempat Alternatif Moda")
    cols = st.columns(4)
    input_data = {"hinc": hinc, "psize": float(psize)}

    for idx, m in enumerate(MODES):
        with cols[idx]:
            st.markdown(f"**Moda {m}**")
            input_data[f"ttme_mode{m}"] = st.number_input(
                f"Waktu Terminal (ttme) - M{m}",
                min_value=0.0, max_value=300.0,
                value=float(default_sample.get(f"ttme_mode{m}", 30.0)), step=1.0
            )
            input_data[f"invc_mode{m}"] = st.number_input(
                f"Biaya Dalam Kendaraan (invc) - M{m}",
                min_value=0.0, max_value=500.0,
                value=float(default_sample.get(f"invc_mode{m}", 30.0)), step=1.0
            )
            input_data[f"invt_mode{m}"] = st.number_input(
                f"Waktu Dalam Kendaraan (invt) - M{m}",
                min_value=0.0, max_value=2000.0,
                value=float(default_sample.get(f"invt_mode{m}", 200.0)), step=5.0
            )
            input_data[f"gc_mode{m}"] = st.number_input(
                f"Biaya Gabungan (gc) - M{m}",
                min_value=0.0, max_value=1000.0,
                value=float(default_sample.get(f"gc_mode{m}", 70.0)), step=1.0
            )

    if st.button("Prediksi Pilihan Moda", type="primary"):
        df_input_raw = pd.DataFrame([input_data])
        df_input_feat = buat_fitur_baru(df_input_raw)

        pred_class = int(pipeline.predict(df_input_feat)[0])
        proba = pipeline.predict_proba(df_input_feat)[0]
        classes = artifact["classes"]

        st.success(f"Hasil Prediksi: Pelancong diprediksi memilih **Moda {pred_class}** (Probabilitas: {max(proba)*100:.2f}%)")

        proba_df = pd.DataFrame({
            "Moda": [f"Moda {c}" for c in classes],
            "Probabilitas": proba
        }).set_index("Moda")

        col_res1, col_res2 = st.columns([1, 2])
        with col_res1:
            st.dataframe(proba_df.style.format({"Probabilitas": "{:.2%}"}), use_container_width=True)
        with col_res2:
            st.bar_chart(proba_df)

with tab_eval:
    st.subheader(f"Perbandingan Cross-Validation & Evaluasi Test ({artifact['best_model_name']})")

    cv_df = pd.DataFrame(artifact["cv_results"]).T
    cv_df.columns = ["CV Accuracy (Mean)", "CV Accuracy (Std)", "CV Macro F1 (Mean)", "CV Macro F1 (Std)"]
    st.markdown("**1. Perbandingan Model pada 5-Fold Stratified Cross-Validation (Data Train)**")
    st.dataframe(cv_df.style.format("{:.4f}"), use_container_width=True)

    test_m = artifact["test_metrics"]
    st.markdown("**2. Performa Evaluasi pada Data Test (1x Evaluasi)**")
    m1, m2, m3 = st.columns(3)
    m1.metric("Test Accuracy", f"{test_m['accuracy']:.2%}")
    m2.metric("Test Macro F1-Score", f"{test_m['f1_macro']:.4f}")
    m3.metric("Test Weighted F1-Score", f"{test_m['f1_weighted']:.4f}")

    st.markdown("**3. Confusion Matrix (Data Test)**")
    cm_df = pd.DataFrame(
        test_m["confusion_matrix"],
        index=[f"Aktual Moda {c}" for c in artifact["classes"]],
        columns=[f"Prediksi Moda {c}" for c in artifact["classes"]]
    )
    st.dataframe(cm_df, use_container_width=True)

Writing app.py


In [8]:
import urllib.request

print("IP Publik Colab (copy angka ini jika diminta):",
      urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip())

!streamlit run app.py &>/content/logs.txt &
!npx localtunnel --port 8501

IP Publik Colab (copy angka ini jika diminta): 34.16.184.88
⠙⠹⠸⠼⠴⠦your url is: https://twenty-eggs-search.loca.lt
^C


In [9]:
%%writefile app.py
import os
import joblib
import pandas as pd
import streamlit as st
from fitur import buat_fitur_baru, MODES

st.set_page_config(page_title="Prediksi Pilihan Moda Transportasi", layout="wide")

MODEL_PATH = os.path.join("models", "model_mode_choice.joblib")

@st.cache_resource
def load_artifact():
    if not os.path.exists(MODEL_PATH):
        return None
    return joblib.load(MODEL_PATH)

artifact = load_artifact()

st.title("Prediksi Pilihan Moda Transportasi (Kasus 9)")
st.markdown(
    "Aplikasi klasifikasi multikelas untuk memprediksi pilihan moda transportasi "
    "(**Moda 1 – 4**) berdasarkan karakteristik perjalanan dan atribut keempat alternatif moda."
)

if artifact is None:
    st.error("File model tidak ditemukan. Jalankan `python latih.py` terlebih dahulu.")
    st.stop()

pipeline = artifact["pipeline"]
default_sample = artifact["default_sample"]

tab_pred, tab_eval = st.tabs(["Simulasi Prediksi", "Metrik & Evaluasi Model"])

with tab_pred:
    st.subheader("1. Profil Pelancong")
    col_ind1, col_ind2 = st.columns(2)
    with col_ind1:
        hinc = st.number_input(
            "Pendapatan Rumah Tangga (hinc)",
            min_value=0.0, max_value=200.0,
            value=float(default_sample.get("hinc", 35.0)), step=1.0
        )
    with col_ind2:
        psize = st.number_input(
            "Ukuran Rombongan (psize)",
            min_value=1, max_value=15,
            value=int(default_sample.get("psize", 1)), step=1
        )

    st.subheader("2. Atribut Keempat Alternatif Moda")
    cols = st.columns(4)
    input_data = {"hinc": hinc, "psize": float(psize)}

    for idx, m in enumerate(MODES):
        with cols[idx]:
            st.markdown(f"**Moda {m}**")
            input_data[f"ttme_mode{m}"] = st.number_input(
                f"Waktu Terminal (ttme) - M{m}",
                min_value=0.0, max_value=300.0,
                value=float(default_sample.get(f"ttme_mode{m}", 30.0)), step=1.0
            )
            input_data[f"invc_mode{m}"] = st.number_input(
                f"Biaya Dalam Kendaraan (invc) - M{m}",
                min_value=0.0, max_value=500.0,
                value=float(default_sample.get(f"invc_mode{m}", 30.0)), step=1.0
            )
            input_data[f"invt_mode{m}"] = st.number_input(
                f"Waktu Dalam Kendaraan (invt) - M{m}",
                min_value=0.0, max_value=2000.0,
                value=float(default_sample.get(f"invt_mode{m}", 200.0)), step=5.0
            )
            input_data[f"gc_mode{m}"] = st.number_input(
                f"Biaya Gabungan (gc) - M{m}",
                min_value=0.0, max_value=1000.0,
                value=float(default_sample.get(f"gc_mode{m}", 70.0)), step=1.0
            )

    if st.button("Prediksi Pilihan Moda", type="primary"):
        df_input_raw = pd.DataFrame([input_data])
        df_input_feat = buat_fitur_baru(df_input_raw)
        # Pastikan urutan kolom persis sama seperti saat training
        df_input_feat = df_input_feat[artifact["engineered_features"]]

        pred_class = int(pipeline.predict(df_input_feat)[0])
        proba = pipeline.predict_proba(df_input_feat)[0]
        classes = artifact["classes"]

        st.success(f"Hasil Prediksi: Pelancong diprediksi memilih **Moda {pred_class}** (Probabilitas: {max(proba)*100:.2f}%)")

        proba_df = pd.DataFrame({
            "Moda": [f"Moda {c}" for c in classes],
            "Probabilitas": proba
        }).set_index("Moda")

        col_res1, col_res2 = st.columns([1, 2])
        with col_res1:
            st.dataframe(proba_df.style.format({"Probabilitas": "{:.2%}"}), use_container_width=True)
        with col_res2:
            st.bar_chart(proba_df)

with tab_eval:
    st.subheader(f"Perbandingan Cross-Validation & Evaluasi Test ({artifact['best_model_name']})")

    cv_df = pd.DataFrame(artifact["cv_results"]).T
    cv_df.columns = ["CV Accuracy (Mean)", "CV Accuracy (Std)", "CV Macro F1 (Mean)", "CV Macro F1 (Std)"]
    st.markdown("**1. Perbandingan Model pada 5-Fold Stratified Cross-Validation (Data Train)**")
    st.dataframe(cv_df.style.format("{:.4f}"), use_container_width=True)

    test_m = artifact["test_metrics"]
    st.markdown("**2. Performa Evaluasi pada Data Test (1x Evaluasi)**")
    m1, m2, m3 = st.columns(3)
    m1.metric("Test Accuracy", f"{test_m['accuracy']:.2%}")
    m2.metric("Test Macro F1-Score", f"{test_m['f1_macro']:.4f}")
    m3.metric("Test Weighted F1-Score", f"{test_m['f1_weighted']:.4f}")

    st.markdown("**3. Confusion Matrix (Data Test)**")
    cm_df = pd.DataFrame(
        test_m["confusion_matrix"],
        index=[f"Aktual Moda {c}" for c in artifact["classes"]],
        columns=[f"Prediksi Moda {c}" for c in artifact["classes"]]
    )
    st.dataframe(cm_df, use_container_width=True)

Overwriting app.py


In [10]:
!pkill -f streamlit || true
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb >/dev/null

!streamlit run app.py --server.port 8501 --server.enableCORS false --server.enableXsrfProtection false &>/content/logs.txt &
!sleep 3
!cloudflared tunnel --url http://localhost:8501

^C
2026-11-17T07:19:38Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-11-17T07:19:38Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-11-17T07:19:42Z INF +--------------------------------------------------------------------------------------------+
2026-11-17T07:19:42Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-11-17T07:19:42Z INF |  https://guy-developed-wed-movers.trycloudflare.com

In [11]:
!mkdir -p dataset
!mv -f kasus09_mode_choice.csv dataset/ 2>/dev/null || true
!zip -r UTS_Kasus09_ModeChoice.zip dataset/ models/ fitur.py latih.py app.py

from google.colab import files
files.download("UTS_Kasus09_ModeChoice.zip")

  adding: dataset/ (stored 0%)
  adding: dataset/kasus09_mode_choice.csv (deflated 79%)
  adding: models/ (stored 0%)
  adding: models/model_mode_choice.joblib (deflated 83%)
  adding: fitur.py (deflated 60%)
  adding: latih.py (deflated 63%)
  adding: app.py (deflated 64%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>